# 序列逆置 （加注意力的seq2seq）
使用attentive sequence to sequence 模型将一个字符串序列逆置。例如 `OIMESIQFIQ` 逆置成 `QIFQISEMIO`(下图来自网络，是一个加attentino的sequence to sequence 模型示意图)
![attentive seq2seq](./seq2seq-attn.jpg)

In [67]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets
import os,sys,tqdm

## 玩具序列数据生成
生成只包含[A-Z]的字符串，并且将encoder输入以及decoder输入以及decoder输出准备好（转成index）

In [68]:
import random
import string

def randomString(stringLength):
    """Generate a random string with the combination of lowercase and uppercase letters """

    letters = string.ascii_uppercase
    return ''.join(random.choice(letters) for i in range(stringLength))

def get_batch(batch_size, length):
    batched_examples = [randomString(length) for i in range(batch_size)]
    enc_x = [[ord(ch)-ord('A')+1 for ch in list(exp)] for exp in batched_examples]
    y = [[o for o in reversed(e_idx)] for e_idx in enc_x]
    dec_x = [[0]+e_idx[:-1] for e_idx in y]
    return (batched_examples, tf.constant(enc_x, dtype=tf.int32), 
            tf.constant(dec_x, dtype=tf.int32), tf.constant(y, dtype=tf.int32))
print(get_batch(2, 10))

(['UIKNULFMMS', 'ZZKKJYFKYP'], <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[21,  9, 11, 14, 21, 12,  6, 13, 13, 19],
       [26, 26, 11, 11, 10, 25,  6, 11, 25, 16]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[ 0, 19, 13, 13,  6, 12, 21, 14, 11,  9],
       [ 0, 16, 25, 11,  6, 25, 10, 11, 11, 26]], dtype=int32)>, <tf.Tensor: shape=(2, 10), dtype=int32, numpy=
array([[19, 13, 13,  6, 12, 21, 14, 11,  9, 21],
       [16, 25, 11,  6, 25, 10, 11, 11, 26, 26]], dtype=int32)>)


# 建立sequence to sequence 模型

完成两空，模型搭建以及单步解码逻辑

In [ ]:
class mySeq2SeqModel(keras.Model):
    def __init__(self):
        super(mySeq2SeqModel, self).__init__()
        self.v_sz=27
        self.hidden = 128
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 64)
        
        self.encoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        self.decoder_cell = tf.keras.layers.SimpleRNNCell(self.hidden)
        
        self.encoder = tf.keras.layers.RNN(self.encoder_cell, 
                                           return_sequences=True, return_state=True)
        self.decoder = tf.keras.layers.RNN(self.decoder_cell, 
                                           return_sequences=True, return_state=True)
        self.dense_attn = tf.keras.layers.Dense(self.hidden)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
        
    # @tf.function
    def call(self, enc_ids, dec_ids):
        '''todo
        完成带attention机制的 sequence2sequence 模型的搭建，模块已经在`__init__`函数中定义好，
        用双线性attention，或者自己改一下`__init__`函数做加性attention
        '''
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb) # enc_out shape(b_sz, len, h_sz), enc_state shape(b_sz, h_sz)
        
        dec_emb = self.embed_layer(dec_ids) # shape(b_sz, len, emb_sz)
        dec_out, _ = self.decoder(dec_emb, initial_state=enc_state) # dec_out shape(b_sz, len, h_sz)
        
        # shape(b_sz, len_dec, len_enc)，计算解码器输出和编码器输出之间的注意力权重，表示解码器在每个时间步应该关注编码器输出的哪些部分
        #att = tf.matmul(self.dense_attn(dec_out), enc_out, transpose_b=True) # shape(b_sz, len_dec, len_enc)
        # shape(b_sz, len_dec, len_enc)，对每个解码器时间步的注意力权重进行归一化，使其和为1，表示解码器在每个时间步应该关注编码器输出的哪些部分
        #att_weight = tf.nn.softmax(att, axis=-1) 
        
        # 采用缩放点积注意力，除以dk的平方根，防止点积过大导致softmax函数的梯度消失
        dk = tf.cast(self.hidden, tf.float32)
        score = tf.matmul(self.dense_attn(dec_out), enc_out, transpose_b=True) / tf.math.sqrt(dk)
        att_weight = tf.nn.softmax(score, axis=-1)

        # shape(b_sz, len_dec, h_sz) ，context是对编码器输出的加权求和，权重由注意力机制计算得到，表示解码器在每个时间步应该关注编码器输出的哪些部分
        context = tf.matmul(att_weight, enc_out) 
        # shape(b_sz, len_dec, h_sz*2)，将解码器的输出和context进行拼接，作为最终的解码器输出，输入到全连接层进行预测
        concat_out = tf.concat([dec_out, context], axis=-1)
        
        logits = self.dense(concat_out) # 输出层，输出每个时间步的预测结果，形状为(batch_size, seq_len, vocab_size)
        return logits
    
    
    @tf.function
    def encode(self, enc_ids):
        enc_emb = self.embed_layer(enc_ids) # shape(b_sz, len, emb_sz)
        enc_out, enc_state = self.encoder(enc_emb)
        return enc_out, [enc_out[:, -1, :], enc_state]
    
    def get_next_token(self, x, state, enc_out):
        '''
        shape(x) = [b_sz,] 
        x是当前时间步的输入，state是当前时间步的状态
        enc_out是编码器的所有输出[b_sz, len_enc, h_sz]
        获取下一个时间步的预测结果，输入为当前时间步的输入和状态，输出为下一个时间步的预测结果和状态
        
        todo
        参考sequence_reversal-exercise, 自己构建单步解码逻辑
        '''
        inp_emb = self.embed_layer(tf.expand_dims(x, axis=-1)) #shape(b_sz, emb_sz)编码
        # Squeeze 掉序列维度，因为 Cell 接收的是 (Batch, Feature)
        inp_emb = tf.squeeze(inp_emb, axis=1)

        h, state = self.decoder_cell(inp_emb, state)

        # 3. Attention Step
        # query: [batch, 1, hidden]，将当前时间步的解码器输出h通过一个全连接层映射到与编码器输出相同的维度，作为查询向量
        #query = tf.expand_dims(self.dense_attn(h), axis=1)
        # 计算该步对所有编码器输出的权重
        #score = tf.matmul(query, enc_out, transpose_b=True) # [batch, 1, enc_len]
        #weights = tf.nn.softmax(score, axis=-1) # [batch, 1, enc_len]，权重归一化，使其和为1
        query = tf.expand_dims(self.dense_attn(h), axis=1)
        dk = tf.cast(self.hidden, tf.float32)
        score = tf.matmul(query, enc_out, transpose_b=True) / tf.math.sqrt(dk)
        weights = tf.nn.softmax(score, axis=-1)
        # context: [batch, 1, hidden]
        context = tf.matmul(weights, enc_out)

        combined = tf.concat([tf.expand_dims(h, axis=1), context], axis=-1)

        logits = self.dense(combined) # shape(b_sz, v_sz)
        
        #  token 是一维的 (batch,)，防止下次循环出错
        out = tf.argmax(logits, axis=-1) # logits 是 (batch, 1, 27)
        out = tf.cast(tf.squeeze(out, axis=1), tf.int32)

        return out, state

# Loss函数以及训练逻辑

In [70]:
@tf.function
def compute_loss(logits, labels):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = tf.reduce_mean(losses)
    return losses

@tf.function
def train_one_step(model, optimizer, enc_x, dec_x, y):
    with tf.GradientTape() as tape:
        logits = model(enc_x, dec_x)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(model, optimizer, seqlen):
    loss = 0.0
    accuracy = 0.0
    for step in range(2000):
        batched_examples, enc_x, dec_x, y = get_batch(32, seqlen)
        loss = train_one_step(model, optimizer, enc_x, dec_x, y)
        if step % 500 == 0:
            print('step', step, ': loss', loss.numpy())
    return loss

# 训练迭代

In [71]:
optimizer = optimizers.Adam(0.0005)
model = mySeq2SeqModel()
train(model, optimizer, seqlen=20)

step 0 : loss 3.2962756
step 500 : loss 1.5401566
step 1000 : loss 0.37531418
step 1500 : loss 0.083771124


<tf.Tensor: shape=(), dtype=float32, numpy=0.015528840944170952>

# 测试模型逆置能力
首先要先对输入的一个字符串进行encode，然后在用decoder解码出逆置的字符串

测试阶段跟训练阶段的区别在于，在训练的时候decoder的输入是给定的，而在预测的时候我们需要一步步生成下一步的decoder的输入

In [72]:
def sequence_reversal():
    def decode(init_state, steps, enc_out):
        b_sz = tf.shape(init_state[0])[0]
        cur_token = tf.zeros(shape=[b_sz], dtype=tf.int32)
        state = init_state
        collect = []
        for i in range(steps):
            cur_token, state = model.get_next_token(cur_token, state, enc_out)
            collect.append(tf.expand_dims(cur_token, axis=-1))
        out = tf.concat(collect, axis=-1).numpy()
        out = [''.join([chr(idx+ord('A')-1) for idx in exp]) for exp in out]
        return out
    
    batched_examples, enc_x, _, _ = get_batch(32, 20)
    enc_out, state = model.encode(enc_x)
    return decode(state, enc_x.get_shape()[-1], enc_out), batched_examples

def is_reverse(seq, rev_seq):
    rev_seq_rev = ''.join([i for i in reversed(list(rev_seq))])
    if seq == rev_seq_rev:
        return True
    else:
        return False
print([is_reverse(*item) for item in list(zip(*sequence_reversal()))])
print(list(zip(*sequence_reversal())))

[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
[('IUDWCLOXPBIELBNLXUQK', 'KQUXLNBLEIBPXOLCWDUI'), ('MBASECFAZSFNPRQCFMWF', 'FWMFCQRPNFSZAFCESABM'), ('DFNUYWDOONRRLFPOOQET', 'TEQOOPFLRRNOODWYUNFD'), ('CHPOZJPRTFQHPZAMRAWH', 'HWARMAZPHQFTRPJZOPHC'), ('HXKYYKMAYRBQNVFHZJKR', 'RKJZHFVNQBRYAMKYYKXH'), ('EBZCJXGXXIXIGARKWKNE', 'ENKWKRAGIXIXXGXJCZBE'), ('IUUAWIGNQFMGADTTBUMN', 'NMUBTTDAGMFQNGIWAUUI'), ('MPMXRXYQUJRVAYJQLLRF', 'FRLLQJYAVRJUQYXRXMPM'), ('HLLGKCNYEKYUAJMJAXWF', 'FWXAJMJAUYKEYNCKGLLH'), ('ZTTIIQFFJZYWJSFWIDSD', 'DSDIWFSJWYZJFFQIITTZ'), ('ATRMNDWZCJPQLYPEOLJV', 'VJLOEPYLQPJCZWDNMRTA'), ('DRBMAJLQEUJBQRXOMWJS', 'SJWMOXRQBJUEQLJAMBRD'), ('EFHCBSAMUOPFJGWYXBOA', 'AOBXYWGJFPOUMASBCHFE'), ('FYARXUXKVPGSAOHUSZWZ', 'ZWZSUHOASGPVKXUXRAYF'), ('PJMEBVZJZDHJYYYIBIFJ', 'JFIBIYYYJHDZJZVBEMJP'), ('RUSUDAWVKWGWURKJHBYY', 'YBBHJKRUWGWKVWADUSUR'), ('JMRD